# 2. The regime model

Four questions, one chart or table each:

1. **How many regimes do the data support?** — the sweep table
2. **What are they?** — the regime map and table
3. **How do they connect?** — the transition heatmap
4. **How far ahead is the model still saying anything?** — the mixing curve

Then the current forecast grid. Stage gates two and three are asserted at the end
of their sections.

Run `forecast fit-regimes` and `forecast forecast-now` before this notebook; both
write what it reads. All computation lives in the package.

In [ ]:
from datetime import date

import pandas as pd

from economic_regime_forecasting import pipeline_gates
from economic_regime_forecasting.configuration.registry import load_registries
from economic_regime_forecasting.configuration.run_settings import (
    ARTIFACTS,
    DEFAULT_RUN_SETTINGS,
)
from economic_regime_forecasting.data.cache import ArtifactStore, SeriesCache
from economic_regime_forecasting.data.panel import assemble_point_in_time_panel, load_final_series
from economic_regime_forecasting.features.observation_matrix import build_observation_matrix
from economic_regime_forecasting.models.model_loading import (
    regime_model_from_dictionary,
)
from economic_regime_forecasting.models.regime_forecast import measure_mixing
from economic_regime_forecasting.models.state_labelling import describe_regimes
from economic_regime_forecasting.reporting import figures, tables

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)

settings = DEFAULT_RUN_SETTINGS
registry, indicators = load_registries()
cache = SeriesCache(settings.cache.raw, settings.cache.vintage)
artifacts = ArtifactStore(settings.cache.models)
today = date.today()

matrix = build_observation_matrix(assemble_point_in_time_panel(registry, today, cache), registry)
model = regime_model_from_dictionary(artifacts.read_json(ARTIFACTS.selected_model))
filtered = model.filtered_state_probabilities(matrix.values)
regimes = describe_regimes(model, matrix.values, matrix.transformed.to_numpy())
labels = [item.compact_label for item in regimes]

print(matrix.describe(), "|", model.state_count, "regimes")

## 1. How many regimes?

Persistence and population are hard floors: a state lasting under three months on
average is noise wearing a label, and one holding under five percent of months
cannot support a conditional rate. Among the candidates that clear both, the
held-out likelihood decides, because it asks the out-of-sample question directly.

The one-state row is the null hypothesis. It is not a regime model at all but a
single Gaussian, and it is what the first acceptance gate compares against.

In [ ]:
sweep = artifacts.read_table(ARTIFACTS.state_count_sweep)
tables.sweep_display_table(sweep)

## 2. What are the regimes?

Read in percent a year, so growth, inflation and the interest rate are on
comparable scales. The model was fitted on standardised values and never told
what any state should mean; the names below are read off the fitted parameters
afterwards.

The map places each regime by what it felt like to live through. Bubble area is
the share of months spent there.

In [ ]:
regime_view = tables.regime_display_table(regimes, registry)
regime_view

In [ ]:
figures.plot_regime_map(regime_view)

### When each regime was active

One lane per regime, filled where the model believed the economy was in it, using
only information available at the time. Recessions are shaded behind for
reference; they are never an input.

A lane chart rather than a stacked area: the model is confident almost everywhere,
so a stack sits pinned at one and spends the whole vertical axis saying so.

In [ ]:
recession = load_final_series(registry, cache, ["recession_indicator"])["recession_indicator"]
figures.plot_regime_timeline(matrix.dates, filtered, labels, recession)

## 3. How do the regimes connect?

Rows are where the economy is, columns where it goes next month. The strong
diagonal is the whole story: these are regimes, not weather. It is also exactly
what limits how far ahead the model can see, which is the next section.

In [ ]:
figures.plot_transition_heatmap(model.transition_matrix, labels)

### Gate 2

In [ ]:
gate_two = pipeline_gates.gate_two_from_tables(
    sweep,
    model,
    model.most_likely_state_path(matrix.values),
    len(matrix),
    per_chain_table=(
        artifacts.read_table(ARTIFACTS.state_count_sweep_by_chain)
        if artifacts.has(ARTIFACTS.state_count_sweep_by_chain)
        else None
    ),
)
print(gate_two.describe())

## 4. How far ahead is the model still saying anything?

A transition matrix mixes. The distance between a projected regime distribution
and the model's long-run distribution decays like the second largest eigenvalue
modulus raised to the horizon. Past some point the projection *is* the
unconditional base rate, and presenting it as a prediction would misdescribe it
even if it scored well.

Where the curve crosses the threshold is that point.

In [ ]:
# Every month to twenty years, not five sampled points: the curve is what the chart
# claims to show, and the crossing of the threshold is read off it (debt D11).
mixing = measure_mixing(
    model,
    filtered,
    range(1, 241),
    settings.information_horizon_total_variation_threshold,
)
print(mixing.describe())
mixing.table().round(4)

In [ ]:
figures.plot_mixing(mixing.table(), settings.information_horizon_total_variation_threshold)

## The current forecast grid

Ten indicators by three horizons, as percentages. The evidence behind each number
follows: `effective sample size` is how many months of history stand behind that
particular forecast, and `distance to stationary` is how far the projection still
is from the base rate.

In [ ]:
forecasts = artifacts.read_table(ARTIFACTS.current_forecasts)
tables.forecast_display_table(forecasts)

In [ ]:
forecasts[
    [
        "indicator",
        "horizon_months",
        "probability",
        "composition",
        "effective_sample_size",
        "distance_to_stationary",
    ]
].round(3)

### Gate 3

In [ ]:
print(
    pipeline_gates.gate_three_forecasts(
        forecasts, list(indicators), settings.forecast_horizons_in_months, mixing
    ).describe()
)